# QLoRA Quality Recovery after MQA Conversion (Gemma 4 E2B)

After `conversion/convert_to_mqa.py` averages the K/V heads in full-attention
layers, the model immediately loses ~1-3 pp on standard benchmarks. This notebook
recovers that loss via short QLoRA fine-tuning with self-distillation from the
original Gemma 4 E2B as the teacher.

Cost: ~4-8 h on A100 for 1-2 epochs over 10-30k samples.

Pipeline:
1. Load **original** Gemma 4 E2B as frozen teacher.
2. Load **MQA-converted** checkpoint as student (trainable via LoRA adapters).
3. KL-divergence loss on teacher logits per token, plus standard CE on the
   reference text.
4. Save LoRA adapters + a merged checkpoint for re-conversion to CoreML.

Prereqs:
- Completed MQA conversion at `MQA_PATH` (safetensors + config, may be partial MQA).
- `eagle_corpus.jsonl` (30k) from `download_eagle_corpus.py`, already on Drive.

**This notebook is orthogonal to train_eagle3_draft.ipynb** — MQA recovery is about
restoring base-model quality, not about draft training.

In [ ]:
!pip install -q -U transformers peft accelerate datasets bitsandbytes safetensors

import os, json, random, time, math
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

HF_TOKEN = ""  # paste https://huggingface.co/settings/tokens
assert HF_TOKEN, "Paste HF token"
from huggingface_hub import login; login(token=HF_TOKEN)

from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")

In [ ]:
# ── Config ───────────────────────────────────────
TEACHER_ID   = "google/gemma-4-E2B-it"
STUDENT_PATH = "/content/drive/MyDrive/gemma-4-E2B-it-mqa"     # output of convert_to_mqa.py
CORPUS_PATH  = "/content/drive/MyDrive/eagle_corpus.jsonl"
SAVE_DIR     = "/content/drive/MyDrive/gemma-4-E2B-it-mqa-recovered"
os.makedirs(SAVE_DIR, exist_ok=True)

NUM_SAMPLES  = 20000
SEQ_LEN      = 1024
EPOCHS       = 1
MICRO_BATCH  = 1
GRAD_ACCUM   = 16
LR           = 2e-4
WARMUP       = 500
KD_ALPHA     = 0.5       # weight on teacher KL vs reference CE
KD_T         = 2.0       # temperature for KL
SEED         = 42

import torch, numpy as np
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

In [ ]:
# ── Load teacher (frozen) ────────────────────────
from transformers import AutoTokenizer
try:
    from transformers import Gemma4ForConditionalGeneration as TCls
except Exception:
    from transformers import AutoModelForCausalLM as TCls

print(f"Loading teacher {TEACHER_ID}...")
teacher = TCls.from_pretrained(TEACHER_ID, torch_dtype=torch.float16, device_map="cuda")
teacher.eval()
for p in teacher.parameters(): p.requires_grad = False
tokenizer = AutoTokenizer.from_pretrained(TEACHER_ID)

# Teacher fit in ~5.5 GB fp16. Student will also fit.
print(f"teacher loaded, mem used = {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Load student (MQA-modified) with 4-bit quant + LoRA ─────────
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# The student was saved by convert_to_mqa.py (possibly with a config that doesn't
# match the transformers runtime if --apply-to != 'all'). Force ignore_mismatched_sizes.
print(f"Loading student {STUDENT_PATH}...")
try:
    student = TCls.from_pretrained(
        STUDENT_PATH, quantization_config=bnb, device_map="auto",
        ignore_mismatched_sizes=True,
    )
except Exception as e:
    print(f"  {e}\n  falling back to fp16 load without bitsandbytes")
    student = TCls.from_pretrained(
        STUDENT_PATH, torch_dtype=torch.float16, device_map="cuda",
        ignore_mismatched_sizes=True,
    )
student = prepare_model_for_kbit_training(student)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
student = get_peft_model(student, lora_cfg)
student.print_trainable_parameters()

In [ ]:
# ── Corpus + tokenization ─────────────────────────
print(f"Reading corpus {CORPUS_PATH}...")
texts = []
with open(CORPUS_PATH) as f:
    for line in f: texts.append(json.loads(line)["text"])
random.Random(SEED).shuffle(texts)
texts = texts[:NUM_SAMPLES]
print(f"  using {len(texts)} sequences")

def tokenize(t):
    return tokenizer.encode(t, return_tensors="pt", truncation=True, max_length=SEQ_LEN)

In [ ]:
# ── Training loop ─────────────────────────────────
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import torch.nn.functional as F
from tqdm.auto import tqdm

opt = AdamW([p for p in student.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
total_steps = len(texts) * EPOCHS // (MICRO_BATCH * GRAD_ACCUM)
def lrfn(step):
    if step < WARMUP: return step / max(1, WARMUP)
    p = (step - WARMUP) / max(1, total_steps - WARMUP)
    return 0.5 * (1 + math.cos(math.pi * p))
sched = LambdaLR(opt, lrfn)

def kd_loss(s_logits, t_logits, T):
    s_log = F.log_softmax(s_logits / T, dim=-1)
    t_p   = F.softmax(t_logits / T, dim=-1)
    return F.kl_div(s_log, t_p, reduction="batchmean") * (T * T)

best_val = float("inf"); step = 0; seen = 0; t0 = time.time()
for epoch in range(EPOCHS):
    print(f"\n=== Epoch {epoch+1}/{EPOCHS} ===")
    random.Random(SEED + epoch).shuffle(texts)
    opt.zero_grad(); loss_ema = None
    for text in tqdm(texts, desc=f"ep{epoch+1}"):
        ids = tokenize(text).to("cuda")
        if ids.shape[1] < 32: continue
        with torch.cuda.amp.autocast(dtype=torch.float16):
            # Teacher forward (no grad)
            with torch.no_grad():
                t_out = teacher(input_ids=ids, use_cache=False)
                t_logits = t_out.logits
            # Student forward
            s_out = student(input_ids=ids, use_cache=False)
            s_logits = s_out.logits
            # Loss: CE vs reference tokens + KD vs teacher
            shift_s = s_logits[:, :-1, :].contiguous()
            shift_t = t_logits[:, :-1, :].contiguous()
            shift_labels = ids[:, 1:].contiguous()
            ce = F.cross_entropy(shift_s.view(-1, shift_s.size(-1)), shift_labels.view(-1))
            kd = kd_loss(shift_s, shift_t, KD_T)
            loss = (1 - KD_ALPHA) * ce + KD_ALPHA * kd
        (loss / GRAD_ACCUM).backward()
        seen += 1
        loss_ema = loss.item() if loss_ema is None else 0.98 * loss_ema + 0.02 * loss.item()
        if (seen % GRAD_ACCUM) == 0:
            torch.nn.utils.clip_grad_norm_([p for p in student.parameters() if p.requires_grad], 1.0)
            opt.step(); sched.step(); opt.zero_grad(); step += 1
            if step % 200 == 0:
                print(f"    step {step} loss={loss_ema:.3f} lr={sched.get_last_lr()[0]:.2e}")
        if step > 0 and step % 2000 == 0 and (seen % GRAD_ACCUM) == 0:
            student.save_pretrained(f"{SAVE_DIR}/adapter_step{step}")

print(f"\nTraining done in {(time.time()-t0)/60:.1f} min")
student.save_pretrained(f"{SAVE_DIR}/adapter_final")

In [ ]:
# ── Merge LoRA into student and export for re-conversion ─────
print("Merging LoRA adapters into student...")
merged = student.merge_and_unload()
merged.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved merged checkpoint: {SAVE_DIR}")
print("\nNext: re-run the CoreML conversion pipeline on this checkpoint to")
print("produce MQA-shaped decode chunks, then benchmark on iPhone.")